# 📊 Prévision des Charges d'Impression — Yazaki 2026
### Charges d'Impression mensuellle par Département ·

| Période | Dates |
|---|---|
| Entraînement | 2023-01 → 2025-12 |
| Test | 2026-01 → 2026-05 |
| Prévision | 2026-06 → 2026-12 |

**Modèles candidats** : Holt · Holt-Winters · ARIMA · SARIMA   
**Sélection automatique** par RMSE minimum sur la période de test


## 1. Imports

In [22]:
import os,sys,warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

from statsmodels.tsa.holtwinters   import ExponentialSmoothing
from statsmodels.tsa.seasonal      import seasonal_decompose
from pmdarima                      import auto_arima
from sklearn.metrics               import mean_absolute_error, mean_squared_error

print("OK - Librairies importees")

OK - Librairies importees


## 2. Chargement des données

In [23]:
project_root = Path(os.getenv('YAZAKI_PROJECT_ROOT', Path.cwd()))
if not (project_root / 'etl').exists():
    parent = Path.cwd().parent
    if (parent / 'etl').exists():
        project_root = parent

# Ajouter le projet au chemin de recherche Python
sys.path.insert(0, str(project_root))

from etl.config import get_clean_engine

with get_clean_engine().connect() as conn:
    charges_imp = pd.read_sql("SELECT * FROM ChargesImpression", conn)

print(f"OK - Project root : {project_root}")
print(f"OK - ChargesImpression chargees depuis Yazaki_Clean : {len(charges_imp):,} lignes")
print(charges_imp.head(3))

OK - Project root : c:\Users\habib\Desktop\Yazaki\ETL-Yazaki
OK - ChargesImpression chargees depuis Yazaki_Clean : 9,945 lignes
   ImpressionID NomDepartement CodeDepartement DateImpression TypeImpression  \
0             1          ACHAT             ACH     2023-03-03          A4-NB   
1             2          ACHAT             ACH     2023-03-15     A4-COULEUR   
2             3          ACHAT             ACH     2023-03-17          A4-NB   

   NbPages  CoutUnitaire FormatPapier CouleurImpression  
0      459         0.026           A4     NOIR ET BLANC  
1      406         0.156           A4           COULEUR  
2       38         0.026           A4     NOIR ET BLANC  


## 3. Préparation des séries mensuelles par département

In [24]:
charges_imp['DateImpression'] = pd.to_datetime(charges_imp['DateImpression'])
charges_imp['NomDepartement'] = charges_imp['NomDepartement'].astype(str).str.strip()
charges_imp = charges_imp[charges_imp['NomDepartement'].str.upper() != 'INCONNU']
charges_imp['CoutImp'] = charges_imp['NbPages'] * charges_imp['CoutUnitaire']

imp_par_dept = (
    charges_imp
    .assign(Mois=charges_imp['DateImpression'].dt.to_period('M'))
    .groupby(['Mois', 'NomDepartement'])['CoutImp']
    .sum().unstack(fill_value=0).astype(float)
    # unstack pour avoir les departements en colonnes et les mois en index, fill_value=0 pour les mois sans impression
)
imp_par_dept.index = imp_par_dept.index.to_timestamp()

print(f"OK - {imp_par_dept.shape[1]} departements | {len(imp_par_dept)} mois")
print(f"     {imp_par_dept.index[0].date()} -> {imp_par_dept.index[-1].date()}")
print(f"\nDepartements : {sorted(imp_par_dept.columns.tolist())}")

OK - 16 departements | 41 mois
     2023-01-01 -> 2026-05-01

Departements : ['ACHAT', 'COSEE', 'DIRECTION', 'EHS', 'ENGENIERIE', 'FINANCE', 'IT', 'LOGISTIQUE', 'NYS', 'OLS', 'PLPP', 'PRODUCTION A', 'PRODUCTION B', 'QUALITE', 'RH', 'TD']


## 4. Visualisation exploratoire

In [25]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Charges d'Impression — Apercu mensuel par departement", fontsize=13, fontweight='bold')

# Top 5 departements
top5 = imp_par_dept.sum().nlargest(5).index
for dept in top5:
    axes[0].plot(imp_par_dept.index, imp_par_dept[dept], marker='o', markersize=3, label=dept)
axes[0].set_title("Top 5 departements")
axes[0].legend(fontsize=8); axes[0].set_ylim(bottom=0)
axes[0].grid(True, alpha=0.4); axes[0].tick_params(axis='x', rotation=20, labelsize=8)

# Total mensuel
total = imp_par_dept.sum(axis=1)
axes[1].fill_between(total.index, total.values, alpha=0.25, color='darkorange')
axes[1].plot(total.index, total.values, color='darkorange', linewidth=2)
axes[1].set_title("Total mensuel — tous departements")
axes[1].set_ylim(bottom=0); axes[1].grid(True, alpha=0.4)
axes[1].tick_params(axis='x', rotation=20, labelsize=8)

plt.tight_layout() # pour eviter que les titres et axes ne se chevauchent
plt.savefig('viz_exploratoire_impression.png', dpi=120, bbox_inches='tight')
# bbox_inches() pour s'assurer que tout le contenu du graphique est inclus dans l'image, même les titres et les étiquettes d'axes
plt.show()
print("OK - viz_exploratoire_impression.png")

OK - viz_exploratoire_impression.png


## 5. Analyser les séries

In [26]:
def analyser_serie(y, dates, titre, couleur='darkorange', out_dir='analyses_imp'):
    os.makedirs(out_dir, exist_ok=True)
    serie = pd.Series(y, index=dates)

    fig = plt.figure(figsize=(16, 10))
    fig.suptitle(f"Analyse - {titre}", fontsize=13, fontweight='bold')
    gs  = fig.add_gridspec(2, 3, hspace=0.5, wspace=0.35)

    # Serie brute + tendance 
    ax1 = fig.add_subplot(gs[0, :3])
    ax1.plot(dates, y, color=couleur, linewidth=1.8, label='Serie brute')
    tendance = serie.rolling(window=6, center=True).mean()
    ax1.plot(dates, tendance, color='red', linewidth=2, linestyle='--', label='Tendance (moy. mobile 6m)')
    ax1.set_title('Serie brute + Tendance')
    ax1.set_ylabel('Charge (TND)', fontsize=9)
    ax1.legend(fontsize=8)
    ax1.set_ylim(bottom=0)
    ax1.grid(True, alpha=0.3)
    ax1.tick_params(axis='x', rotation=20, labelsize=8)

    # Decomposition 
    try:
        # extrapolate_trend='freq' estime la tendance sur les bords, sinon on perdrait les 6 premiers et 6 derniers mois
        decomp = seasonal_decompose(serie, model='additive', period=12, extrapolate_trend='freq')
        for ax, data, title, color in [
            (fig.add_subplot(gs[1, 0]), decomp.trend,    'Tendance',     'red'),
            (fig.add_subplot(gs[1, 1]), decomp.seasonal, 'Saisonnalite', 'green'),
            (fig.add_subplot(gs[1, 2]), decomp.resid,    'Residu',       'purple'),
        ]:
            ax.plot(dates, data, color=color, linewidth=1.5)
            if title == 'Residu':
                ax.axhline(0, color='black', linewidth=0.8)
            ax.set_title(title, fontsize=9, fontweight='bold')
            ax.set_ylabel('Valeur', fontsize=8)
            ax.grid(True, alpha=0.3)
            ax.tick_params(axis='x', rotation=20, labelsize=7)
    except Exception as e:
        print(f"  Decomposition non disponible : {e}")

    plt.tight_layout()
    path = os.path.join(out_dir, f"analyse_{titre.replace(' ','_').replace('/','_')}.png")
    plt.savefig(path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Sauvegarde : {path}")
    return path

### Lancement de l'analyse sur tous les départements 

In [27]:
# Analyser tous les departements
print("Analyse tous les departements :") 
for dept in imp_par_dept.sum().index:
    analyser_serie(imp_par_dept[dept].values, imp_par_dept.index, dept)

print("\nOK - Analyses sauvegardees dans analyses_imp/")

Analyse tous les departements :
  ✓ Sauvegarde : analyses_imp\analyse_ACHAT.png
  ✓ Sauvegarde : analyses_imp\analyse_COSEE.png
  ✓ Sauvegarde : analyses_imp\analyse_DIRECTION.png
  ✓ Sauvegarde : analyses_imp\analyse_EHS.png
  ✓ Sauvegarde : analyses_imp\analyse_ENGENIERIE.png
  ✓ Sauvegarde : analyses_imp\analyse_FINANCE.png
  ✓ Sauvegarde : analyses_imp\analyse_IT.png
  ✓ Sauvegarde : analyses_imp\analyse_LOGISTIQUE.png
  ✓ Sauvegarde : analyses_imp\analyse_NYS.png
  ✓ Sauvegarde : analyses_imp\analyse_OLS.png
  ✓ Sauvegarde : analyses_imp\analyse_PLPP.png
  ✓ Sauvegarde : analyses_imp\analyse_PRODUCTION_A.png
  ✓ Sauvegarde : analyses_imp\analyse_PRODUCTION_B.png
  ✓ Sauvegarde : analyses_imp\analyse_QUALITE.png
  ✓ Sauvegarde : analyses_imp\analyse_RH.png
  ✓ Sauvegarde : analyses_imp\analyse_TD.png

OK - Analyses sauvegardees dans analyses_imp/


### Diagnostic de variabilité

In [28]:
# Diagnostic variabilite avant lancement
print("=" * 65)
print("  DIAGNOSTIC VARIABILITE — CHARGES IMPRESSION")
print("=" * 65)
for dept in imp_par_dept.columns:
    serie = imp_par_dept[dept].values.astype(float)
    mean_ = np.mean(serie) # _ veux dire "variable temporaire dont on se sert juste pour calculer une autre variable, ici le CV, et qu'on n'a pas besoin de garder en memoire apres"
    std_  = np.std(serie)
    cv    = std_ / mean_ * 100 if mean_ > 0 else 0
    flag  = "⚠️  TROP PLATE" if cv < 10 else "✅"
    print(f"  {dept:<16} mean={mean_:7.1f}  std={std_:6.1f}  CV={cv:5.1f}%  {flag}")

  DIAGNOSTIC VARIABILITE — CHARGES IMPRESSION
  ACHAT            mean=  399.8  std= 119.2  CV= 29.8%  ✅
  COSEE            mean=  108.1  std=  35.1  CV= 32.5%  ✅
  DIRECTION        mean=   56.0  std=  14.6  CV= 26.0%  ✅
  EHS              mean=   43.9  std=  22.5  CV= 51.3%  ✅
  ENGENIERIE       mean=  255.0  std=  50.3  CV= 19.7%  ✅
  FINANCE          mean=   66.6  std=  75.7  CV=113.6%  ✅
  IT               mean=   41.5  std=  37.4  CV= 90.2%  ✅
  LOGISTIQUE       mean=  120.3  std=  20.9  CV= 17.4%  ✅
  NYS              mean=   80.7  std=  41.1  CV= 51.0%  ✅
  OLS              mean=  129.0  std=  27.6  CV= 21.4%  ✅
  PLPP             mean=  162.8  std=  55.7  CV= 34.2%  ✅
  PRODUCTION A     mean=   70.5  std=  17.6  CV= 25.0%  ✅
  PRODUCTION B     mean=  158.0  std=  28.7  CV= 18.1%  ✅
  QUALITE          mean=  188.9  std=  64.1  CV= 33.9%  ✅
  RH               mean=  137.9  std=  42.5  CV= 30.8%  ✅
  TD               mean=  196.3  std=  63.8  CV= 32.5%  ✅


## 6. Modèles de prévision

| Modèle | Librairie | Quand testé |
|---|---|---|
| **Holt** | statsmodels | Tendance sans saisonnalité |
| **Holt-Winters** | statsmodels | Saisonnalité détectée |
| **SARIMA** | pmdarima | Toujours (s'adapte automatiquement) |

**Sélection** : meilleur RMSE sur la période test (2026-01 → 2026-05)  
**Pré-qualification** : ADF + ACF + CV → seuls les modèles adaptés sont testés


### Paramètres globaux · sMAPE · nom_sarima

In [29]:
TRAIN_START    = pd.Timestamp('2023-01-01')
TRAIN_END      = pd.Timestamp('2025-12-31')
TEST_START     = pd.Timestamp('2026-01-01')
TEST_END       = pd.Timestamp('2026-05-31')
FORECAST_START = pd.Timestamp('2026-06-01')
FORECAST_END   = pd.Timestamp('2026-12-31')

N_TEST       = 5
N_FORECAST   = 7
WF_MIN_TRAIN = 24  # WF = Walk Forward

# sMAPE : Symmetric Mean Absolute Percentage Error
def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask  = denom > 0
    return np.mean(np.abs(y_true[mask] - y_pred[mask]) / denom[mask]) * 100

# Fonction pour nommer les modèles SARIMA
def nom_sarima(m):
    p,d,q   = m.order
    P,D,Q,s = m.seasonal_order
    return f"SARIMA({p},{d},{q})({P},{D},{Q})[{s}]" if any([P,D,Q]) else f"ARIMA({p},{d},{q})"


### Étape 1 — Pré-qualification des modèles (`choisir_modeles`)

In [30]:
# ═══════════════════════════════════════════════════════════════
# ETAPE 1 : PRE-QUALIFICATION
# ═══════════════════════════════════════════════════════════════
def choisir_modeles(y_train):
    from statsmodels.tsa.stattools import adfuller, acf as acf_fn

    n     = len(y_train)
    mean_ = np.mean(y_train)
    cv    = np.std(y_train) / mean_ if mean_ > 0 else 0

    try:
        stationnaire = adfuller(y_train, autolag='AIC', maxlag=12)[1] < 0.05
    except:
        stationnaire = False

    # Seuil 0.10 
    try:
        saisonniere = abs(acf_fn(y_train, nlags=13, fft=True)[12]) > 0.10
    except:
        saisonniere = False

    modeles = []

    # Holt : tendance sans saisonnalite
    if not saisonniere and cv >= 0.05:
        modeles.append('holt')

    # Holt-Winters : saisonnalite OU CV >= 0.15 + assez de donnees
    # CV >= 0.15 suffit car HW peut capter des patterns meme sans saisonnalite stricte
    if n >= 24 and (saisonniere or cv >= 0.15):
        modeles.append('hw')

    # SARIMA : toujours
    modeles.append('arima')


    if not modeles: 
        modeles = ['arima']

    return modeles, {
        'stationnaire': stationnaire,
        'saisonniere' : saisonniere,
        'cv'          : round(cv, 3),
    }


### Étape 2 — Entraînement & prédiction (`entrainer_predire`)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Étape 2 : ENTRAINER + PREDIRE
# ═══════════════════════════════════════════════════════════════
def entrainer_predire(mtype, y_train, n_steps, dates_train=None):
    try:
        if mtype == 'holt':
            m = ExponentialSmoothing(
                    y_train, trend='add', seasonal=None,
                    initialization_method='estimated').fit(optimized=True)
            return np.clip(m.forecast(n_steps), 0, None) 

        elif mtype == 'hw':
            m = ExponentialSmoothing(
                    y_train, trend='add', seasonal='add',
                    seasonal_periods=12,
                    initialization_method='estimated').fit(optimized=True)
            return np.clip(m.forecast(n_steps), 0, None) # clip pour eviter les previsions negatives qui n'ont pas de sens pour des couts

        elif mtype == 'arima':
            m = auto_arima(
                    y_train, seasonal=True, m=12, stepwise=True,
                    suppress_warnings=True, error_action='ignore',
                    max_p=3, max_q=3, max_P=2, max_Q=2,
                    d=1, max_d=2, max_D=1,
                    information_criterion='aic', with_intercept=True, trend='c')
            return np.clip(m.predict(n_periods=n_steps), 0, None)

    except:
        return None


### Étape 3 — Fitted values pour visualisation (`calculer_fitted`)

In [32]:
# ═══════════════════════════════════════════════════════════════
# ETAPE 3 : FITTED VALUES pour visualisation
# ═══════════════════════════════════════════════════════════════
def calculer_fitted(mtype, y_full, dates_full, n_forecast):
    try:
        if mtype == 'holt':
            mf = ExponentialSmoothing(
                    y_full, trend='add', seasonal=None,
                    initialization_method='estimated').fit(optimized=True)
            return mf.fittedvalues.values

        elif mtype == 'hw':
            mf = ExponentialSmoothing(
                    y_full, trend='add', seasonal='add', seasonal_periods=12,
                    initialization_method='estimated').fit(optimized=True)
            return mf.fittedvalues.values

        elif mtype == 'arima':
            mf = auto_arima(
                    y_full, seasonal=True, m=12, stepwise=True,
                    suppress_warnings=True, error_action='ignore',
                    max_p=3, max_q=3, max_P=2, max_Q=2,
                    d=1, max_d=2, max_D=1,
                    information_criterion='aic', with_intercept=True, trend='c')
            return y_full - mf.resid() # mf.resid() = fittedvalues

    except:
        return np.full(len(y_full), np.mean(y_full))


### Étape 4 — Pipeline principal (`prevoir_departement`)

In [33]:
# ═══════════════════════════════════════════════════════════════
# Étape 4 : PIPELINE PRINCIPAL
# ═══════════════════════════════════════════════════════════════
def prevoir_departement(nom_dept, y_full, dates_full,
                        n_forecast=N_FORECAST, n_test=N_TEST):

    y_train = y_full[:-n_test]
    y_test  = y_full[-n_test:]
    d_train = dates_full[:-n_test]

    # ── Etape 1 : Pre-qualification ──────────────────────────────
    modeles, infos = choisir_modeles(y_train)
    print(f"  [{nom_dept}]  stat={infos['stationnaire']}  "
          f"sais={infos['saisonniere']}  CV={infos['cv']:.2f}  "
          f"candidats={modeles}")

    # ── Etape 2 : Evaluer chaque candidat sur la période test ────
    scores = {}
    for mtype in modeles:
        pred = entrainer_predire(mtype, y_train, n_test, d_train)
        if pred is not None:
            rmse = np.sqrt(mean_squared_error(y_test, pred))
            scores[mtype] = {'rmse': rmse, 'pred': pred}

    if not scores:
        print(f"    Aucun modele n'a converge -> fallback arima")
        pred_fallback = np.full(n_test, np.mean(y_train[-6:]))
        scores['arima'] = {'rmse': np.inf, 'pred': pred_fallback}

    # Choisir le modele avec le RMSE le plus faible
    meilleur  = min(scores, key=lambda m: scores[m]['rmse'])
    pred_test = scores[meilleur]['pred']

    print(f"    Evaluation test (n={n_test}) :")
    for m, s in sorted(scores.items(), key=lambda x: x[1]['rmse']):
        tag = '  <-- CHOISI' if m == meilleur else ''
        print(f"      {m:<35} RMSE={s['rmse']:7.2f}{tag}")

    # ── Etape 3 : Métriques finales sur la période test ──────────
    rmse_test  = np.sqrt(mean_squared_error(y_test, pred_test))
    mae_test   = mean_absolute_error(y_test, pred_test)
    smape_test = smape(y_test, pred_test)
    print(f"    Test fixe  RMSE={rmse_test:.2f}  "
          f"MAE={mae_test:.2f}  sMAPE={smape_test:.1f}%")

    # ── Etape 4 : Forecast final (reentrainer sur y_full) ────────
    forecast = entrainer_predire(meilleur, y_full, n_forecast, dates_full)
    if forecast is None:
        forecast = np.full(n_forecast, np.mean(y_full[-6:]))

    fitted = calculer_fitted(meilleur, y_full, dates_full, n_forecast)

    future_dates = pd.date_range(
        start=dates_full[-1] + pd.DateOffset(months=1),
        periods=n_forecast, freq='MS')

    # Tableau comparatif 
    comparatif = pd.DataFrame([{
        'Modele'     : m,
        'RMSE_test'  : round(s['rmse'], 2),
        'Selectionne': 'OUI' if m == meilleur else '',
    } for m, s in scores.items()]).sort_values('RMSE_test').reset_index(drop=True)

    print(f"  OK [{nom_dept:<16}]  {meilleur:<35}  "
          f"RMSE={rmse_test:.2f}  sMAPE={smape_test:.1f}%")

    return {
        'dept'        : nom_dept,
        'modele'      : meilleur,
        'rmse'        : rmse_test,
        'mae'         : mae_test,
        'smape'       : smape_test,
        'comparatif'  : comparatif,
        'forecast'    : forecast,
        'future_dates': future_dates,
        'fitted'      : fitted,
        'pred_test'   : pred_test,
        'y_test'      : y_test,
        'dates'       : dates_full,
        'y'           : y_full,
    }

print("OK - Pipeline charge")
print("  Modeles           : Holt / Holt-Winters / SARIMA")
print("  Selection         : RMSE min sur periode test fixe (2026-01->2026-05)")
print("  ARIMA             : d=1 force")
print("  Forecast          : 2026-06 -> 2026-12 (7 mois)")

OK - Pipeline charge
  Modeles           : Holt / Holt-Winters / SARIMA
  Selection         : RMSE min sur periode test fixe (2026-01->2026-05)
  ARIMA             : d=1 force
  Forecast          : 2026-06 -> 2026-12 (7 mois)


## 7. Lancer les prévisions

In [34]:
print("=" * 65)
print("  CHARGES IMPRESSION — Prevision par departement")
print("=" * 65)

resultats_imp = {}
for dept in imp_par_dept.columns:
    serie = imp_par_dept[dept].astype(float).loc[
        (imp_par_dept.index >= TRAIN_START) & (imp_par_dept.index <= TEST_END)
    ]
    if len(serie) <= N_TEST:
        print(f"  [{dept}] serie trop courte — ignoree")
        continue
    r = prevoir_departement(dept, serie.values, serie.index,
                            n_forecast=N_FORECAST, n_test=N_TEST)
    if r:
        resultats_imp[dept] = r

print(f"\n  => {len(resultats_imp)} departement(s) traite(s)")

  CHARGES IMPRESSION — Prevision par departement
  [ACHAT]  stat=True  sais=True  CV=0.29  candidats=['hw', 'arima']
    Evaluation test (n=5) :
      arima                               RMSE= 113.71  <-- CHOISI
      hw                                  RMSE= 142.55
    Test fixe  RMSE=113.71  MAE=89.72  sMAPE=29.0%
  OK [ACHAT           ]  arima                                RMSE=113.71  sMAPE=29.0%
  [COSEE]  stat=False  sais=True  CV=0.30  candidats=['hw', 'arima']
    Evaluation test (n=5) :
      hw                                  RMSE=  43.76  <-- CHOISI
      arima                               RMSE=  55.08
    Test fixe  RMSE=43.76  MAE=33.71  sMAPE=45.3%
  OK [COSEE           ]  hw                                   RMSE=43.76  sMAPE=45.3%
  [DIRECTION]  stat=True  sais=True  CV=0.27  candidats=['hw', 'arima']
    Evaluation test (n=5) :
      arima                               RMSE=   7.78  <-- CHOISI
      hw                                  RMSE=  14.27
    Test fixe  RMS

## 8. Tableau des performances

In [35]:
def tableau_performances(resultats, label):
    print(f"\n{'='*65}")
    print(f"  {label} — Resume global (trie par RMSE)")
    print(f"{'='*65}")
    rows = [{'Departement': dept, 'Meilleur Modele': r['modele'],
             'RMSE': round(r['rmse'], 2), 'MAE': round(r['mae'], 2),
             'sMAPE%': round(r['smape'], 1)}
            for dept, r in resultats.items()]
    df = pd.DataFrame(rows).set_index('Departement').sort_values('RMSE')
    print(df.to_string())
    print(f"\n  sMAPE moyen : {df['sMAPE%'].mean():.1f}%")
    print(f"\n  Comparatif modeles par departement :")
    for dept, r in resultats.items():
        print(f"\n  [{dept}]")
        print(r['comparatif'].to_string(index=False))
    return df

recap_imp = tableau_performances(resultats_imp, "Charges Impression")


  Charges Impression — Resume global (trie par RMSE)
             Meilleur Modele    RMSE     MAE  sMAPE%
Departement                                         
DIRECTION              arima    7.78    6.88    11.4
LOGISTIQUE                hw   14.46   12.00     9.2
ENGENIERIE                hw   15.08   12.54     4.5
PRODUCTION A              hw   18.92   16.39    27.5
EHS                     holt   19.40   14.54    26.6
OLS                    arima   28.34   25.18    23.5
PRODUCTION B           arima   29.14   26.56    17.1
NYS                       hw   33.30   27.50    41.1
IT                      holt   41.56   37.50    97.6
COSEE                     hw   43.76   33.71    45.3
RH                     arima   47.42   31.69    25.7
TD                      holt   69.62   53.05    37.4
QUALITE                arima   71.35   43.56    46.5
PLPP                    holt   73.98   70.31    48.3
ACHAT                  arima  113.71   89.72    29.0
FINANCE                   hw  133.63  108.00 

## 9. Visualisations avancées — Réel vs Prévisions

In [40]:
def comparaison_reel_prevision(resultats, label, c_reel, c_prev, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    depts = sorted(resultats.keys())
    
    # Create a separate visualization for each department
    for dept in depts:
        r = resultats[dept]
        
        fig, ax = plt.subplots(1, 1, figsize=(10, 6))
        fig.suptitle(f'{label} — {dept}\n{r["modele"]} | sMAPE={r["smape"]:.1f}%', 
                     fontsize=13, fontweight='bold')
        
        n_test    = len(r['y_test'])
        n_fc      = len(r['forecast'])
        x_test    = np.arange(n_test)
        x_future  = np.arange(n_test, n_test + n_fc)

        ax.bar(x_test - 0.2, r['y_test'],    width=0.4, label='Reel (test)', color=c_reel, alpha=0.8)
        ax.bar(x_test + 0.2, r['pred_test'], width=0.4, label='Predit (test)', color=c_prev, alpha=0.8)
        ax.bar(x_future, r['forecast'], width=0.6, color=c_prev, alpha=0.3,
               edgecolor=c_prev, linewidth=1.5, label=f'Forecast ({n_fc}m)')

        ax.axvline(n_test - 0.5, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
        ax.set_ylabel('Charge (TND)', fontsize=11)
        ax.set_xlabel('Période', fontsize=11)

        labels = [f'T{i+1}' for i in range(n_test)] + [f'P{i+1}' for i in range(n_fc)]
        ax.set_xticks(np.arange(n_test + n_fc))
        ax.set_xticklabels(labels, fontsize=9, rotation=45)
        ax.grid(True, alpha=0.2, axis='y')
        ax.legend(fontsize=10, loc='upper left')

        plt.tight_layout()
        
        # Save individual file for each department
        dept_clean = dept.replace(' ', '_').replace('/', '_')
        path = os.path.join(out_dir, f'reel_vs_previsions_{dept_clean}.png')
        plt.savefig(path, dpi=120, bbox_inches='tight')
        plt.close()
        print(f"OK - {path}")

comparaison_reel_prevision(resultats_imp, "Charges Impression",
                           c_reel='darkorange', c_prev='#7c3aed',
                           out_dir='previsions_imp')

OK - previsions_imp\reel_vs_previsions_ACHAT.png
OK - previsions_imp\reel_vs_previsions_COSEE.png
OK - previsions_imp\reel_vs_previsions_DIRECTION.png
OK - previsions_imp\reel_vs_previsions_EHS.png
OK - previsions_imp\reel_vs_previsions_ENGENIERIE.png
OK - previsions_imp\reel_vs_previsions_FINANCE.png
OK - previsions_imp\reel_vs_previsions_IT.png
OK - previsions_imp\reel_vs_previsions_LOGISTIQUE.png
OK - previsions_imp\reel_vs_previsions_NYS.png
OK - previsions_imp\reel_vs_previsions_OLS.png
OK - previsions_imp\reel_vs_previsions_PLPP.png
OK - previsions_imp\reel_vs_previsions_PRODUCTION_A.png
OK - previsions_imp\reel_vs_previsions_PRODUCTION_B.png
OK - previsions_imp\reel_vs_previsions_QUALITE.png
OK - previsions_imp\reel_vs_previsions_RH.png
OK - previsions_imp\reel_vs_previsions_TD.png


## 10. Enregistrement SQL Server + Export CSV

In [37]:
from sqlalchemy import text
from etl.config import get_dw_engine

def sauvegarder_previsions_sql(resultats, charge_type, date_min=TRAIN_START, date_max=FORECAST_END):
    engine = get_dw_engine()
    rows   = []
    test_dates = pd.date_range(start=TEST_START, end=TEST_END, freq='MS')

    for dept, r in resultats.items():
        for d, v in zip(r['dates'], r['y']):
            rows.append({'NomDepartement': dept, 'Mois': pd.to_datetime(d).date(),
                         'Type': 'Historique', 'ChargeType': charge_type,
                         'ChargeValue': round(float(v), 2), 'Modele': None})

        for d, v in zip(test_dates, r['pred_test']):
            rows.append({'NomDepartement': dept, 'Mois': pd.to_datetime(d).date(),
                         'Type': 'Prevision', 'ChargeType': charge_type,
                         'ChargeValue': round(float(v), 2), 'Modele': str(r.get('modele', ''))})

        for d, v in zip(r['future_dates'], r['forecast']):
            rows.append({'NomDepartement': dept, 'Mois': pd.to_datetime(d).date(),
                         'Type': 'Prevision', 'ChargeType': charge_type,
                         'ChargeValue': round(float(v), 2), 'Modele': str(r.get('modele', ''))})

    if not rows:
        print(f"Aucune ligne a inserer pour {charge_type}")
        return 0

    df_rows = pd.DataFrame(rows)
    with engine.begin() as conn:
        dim_dept = pd.read_sql("SELECT DepartementID, NomDepartement FROM Dim_Departement", conn)
        dept_map = dict(zip(dim_dept['NomDepartement'], dim_dept['DepartementID']))
        df_rows['DepartementID'] = df_rows['NomDepartement'].map(dept_map)

        manquants = sorted(df_rows[df_rows['DepartementID'].isna()]['NomDepartement'].unique().tolist())
        if manquants:
            print(f"Departements absents dans Dim_Departement (ignores) : {manquants}")

        df_rows = df_rows.dropna(subset=['DepartementID']).copy()
        if df_rows.empty:
            print(f"Aucune ligne valide pour {charge_type}")
            return 0

        conn.execute(text("""
            DELETE FROM Previsions
            WHERE ChargeType = :charge_type
              AND Mois BETWEEN :date_min AND :date_max
        """), {'charge_type': charge_type,
                 'date_min': pd.to_datetime(date_min).date(),
                 'date_max': pd.to_datetime(date_max).date()})

        payload = [{'DepartementID': int(row.DepartementID), 'Mois': row.Mois,
                    'Type': row.Type, 'ChargeType': row.ChargeType,
                    'ChargeValue': float(row.ChargeValue),
                    'Modele': (None if pd.isna(row.Modele) or row.Modele == ''
                               else str(row.Modele))}
                   for row in df_rows.itertuples(index=False)]

        conn.execute(text("""
            INSERT INTO Previsions (DepartementID, Mois, Type, ChargeType, ChargeValue, Modele)
            VALUES (:DepartementID, :Mois, :Type, :ChargeType, :ChargeValue, :Modele)
        """), payload)

    print(f"OK - {len(payload)} ligne(s) inseree(s) dans Previsions pour {charge_type}")
    return len(payload)


### Lancement de l'enregistrement SQL

In [38]:
print("=" * 65)
print("  ENREGISTREMENT SQL SERVER — TABLE Previsions")
print("=" * 65)
n_imp = sauvegarder_previsions_sql(resultats_imp, 'Impression')
print(f"Total insere : {n_imp} ligne(s)")

  ENREGISTREMENT SQL SERVER — TABLE Previsions
OK - 848 ligne(s) inseree(s) dans Previsions pour Impression
Total insere : 848 ligne(s)


In [39]:
def exporter(resultats, fichier):
    rows = []
    test_dates = pd.date_range(start=TEST_START, end=TEST_END, freq='MS')
    for dept, r in resultats.items():
        for d, v in zip(r['dates'], r['y']):
            rows.append({'Departement': dept, 'Mois': d.date(), 'Type': 'Historique',
                         'Charge_TND': round(float(v), 2), 'Modele': ''})
        for d, v in zip(test_dates, r['pred_test']):
            rows.append({'Departement': dept, 'Mois': d.date(), 'Type': 'Prevision_Test',
                         'Charge_TND': round(float(v), 2), 'Modele': r['modele']})
        for d, v in zip(r['future_dates'], r['forecast']):
            rows.append({'Departement': dept, 'Mois': d.date(), 'Type': 'Prevision',
                         'Charge_TND': round(float(v), 2), 'Modele': r['modele']})

    df_export = pd.DataFrame(rows).sort_values(['Departement', 'Mois', 'Type'])
    try:
        df_export.to_csv(fichier, index=False)
        print(f"OK - {fichier}")
        return fichier
    except PermissionError:
        p = Path(fichier)
        fallback = p.with_name(f"{p.stem}_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}{p.suffix}")
        df_export.to_csv(fallback, index=False)
        print(f"WARNING - Fichier verrouille. Export alternatif : {fallback}")
        return str(fallback)

f_imp = exporter(resultats_imp, 'previsions_impression.csv')
print(f"\nFichiers generes :")
print(f"  {f_imp}")
print(f"  previsions_imp/previsions.png")
print(f"  previsions_imp/reel_vs_previsions.png")
print(f"\nTypes disponibles pour Power BI : Historique / Prevision_Test / Prevision")

OK - previsions_impression.csv

Fichiers generes :
  previsions_impression.csv
  previsions_imp/previsions.png
  previsions_imp/reel_vs_previsions.png

Types disponibles pour Power BI : Historique / Prevision_Test / Prevision
